- Read QuakeML
- Filter catalog
- Define template parameters
- Download miniseeds surrounding the templates
- Create template objects from miniseeds, save
- Perform detection using template matching

In [49]:
import obspy
import eqcorrscan
import numpy as np
import pandas as pd
import tarfile

### Define network parameters

In [50]:
stations = ['KEMF','NCHR','ENWF','KEMO']
network = 'NV'

### Define template parameters

In [51]:
lowcut = 8.0
highcut = 35.0
filt_order = 4
data_pad = 20.
samp_rate = 200.0
length = 0.5
prepick = 0.05
process_len = 86400
min_snr = 0.1
swin = 'all'

### Define starting catalog filters

In [52]:
min_lat = 47.9
max_lat = 48.05
min_lon = -129.15
max_lon = -129.05
min_time = obspy.UTCDateTime(2017,6,1,1,30,0)
max_time = obspy.UTCDateTime(2017,7,1,1,30,0)
min_magnitude = 1.2
max_magnitude = 'NaN'

### Define detection parameters

In [53]:
threshold = 0.5
threshold_type = 'av_chan_corr'
trig_int = 1
# etc.

### Define pathnames

In [54]:
# Path to starting catalog in QuakeML format:
starting_cat_path = 'endeavour_june2017.xml'
# Path to miniseed waveform data in tar format:
data_path = "/data/wsd01/endeavor.tar.gz"

### Read in starting earthquake catalog, in QuakeML format
To-Do: starting catalog needs to have event IDs and phase hints. Will need to make a new catalog.

In [55]:
cat = obspy.core.event.catalog.read_events(starting_cat_path)

#### Filter catalog

In [56]:
min_time = min_time.datetime.strftime('%Y-%m-%dT%H:%M:%S')
max_time = max_time.datetime.strftime('%Y-%m-%dT%H:%M:%S')
t1_filter = "time > " + min_time
t2_filter = "time < " + max_time
lon1_filter = "longitude > " + str(min_lon)
lon2_filter = "longitude < " + str(max_lon)
lat1_filter = "latitude > " + str(min_lat)
lat2_filter = "latitude < " + str(max_lat)
mag1_filter = "magnitude > " + str(min_magnitude)
mag2_filter = "magnitude < " + str(max_magnitude)
cat = cat.filter(t1_filter,t2_filter,lon1_filter,lon2_filter,lat1_filter,lat2_filter,mag1_filter)

In [68]:
# Add in a phase hint to the picks
for event in cat.events:
    for pick in event.picks:
        pick_id = pick.resource_id
        arr = [a for a in event.origins[0].arrivals if a.pick_id==pick_id]
        pick.phase_hint=arr[0].phase

Event:	2017-06-01T20:05:38.782000Z | +48.028, -129.070 | 1.24 Mw

	 event_type: 'earthquake'
	       ---------
	      picks: 6 Elements
	    origins: 1 Elements
	 magnitudes: 1 Elements

In [67]:
# Band-aid for the time being
for ev in cat:
    print(ev.magnitudes[0].resource_id.id[-6:])
    ev.resource_id.id = str(ev.magnitudes[0].resource_id.id[-6:])
    for p in ev.picks:
        if p.waveform_id.channel_code[-1]=='Z':
            p.phase_hint = 'P'
        else:
            p.phase_hint = 'S'

515146


AttributeError: 'NoneType' object has no attribute 'id'

### Download waveform data for the catalog duration

In [6]:
# Get list of stations and channels from QuakeML
picks = [p.picks for p in cat.events]
picks = sum(picks,[])

networks = [p.waveform_id.network_code for p in picks]
stations = [p.waveform_id.station_code for p in picks]
channels = [p.waveform_id.channel_code[0:2] + '*' for p in picks]
# Toss pressure channels:
channelToRemove = 'HD*'
channels = [value for value in channels if value != channelToRemove]

sta_list = [f"{n}.{s}..{c[0:2]}" for n, s, c in zip(networks,stations,channels)]
sta_list = np.unique(sta_list)

# Get all the info for those stations from IRIS
network = ",".join((np.unique(networks)).tolist())
channel = ",".join((np.unique(channels)).tolist())
station = ",".join((np.unique(stations)).tolist())


## Miniseed naming conventions:
KEMF.NV.2017.yearday

In [14]:
# Loop over days and download each (unprocessed) into a directory
path = 'data_june2017/'
t1 = obspy.UTCDateTime(2017,6,1)
t2 = obspy.UTCDateTime(2017,6,3)

client = obspy.clients.fdsn.client.Client('IRIS')

time_bins = np.arange(t1.datetime,t2.datetime,pd.Timedelta(1,'days'))
time_bins = [pd.to_datetime(t) for t in time_bins]

for t in time_bins:
    
    t1 = obspy.UTCDateTime(t)
    t2 = obspy.UTCDateTime(t + pd.Timedelta(1,'day'))
    
    pathname = path + t1.strftime("%Y%m%d")+'.mseed'
    
    st = client.get_waveforms(network,station,'',channel,t1,t2)
    
    st.write(pathname)

## Link to the already downloaded data

In [33]:
tar = tarfile.open(data_path, "r")

### Create templates from miniseeds
TO-DO: this should be rewritten to loop over days, so that we aren't pulling in a new stream every time

In [47]:
# Start base day as none
base_day = None

# Loop over events in catalog and make a template for each
# Load one day-long stream in as needed
for i,ev in enumerate(cat):
    
    # Check if we need to pull in a new stream
    if ev.origins[0].time.datetime.strftime('%Y%m%d') != base_day:
        # Reset the day we are on and pull in a new stream
        base_day = ev.origins[0].time.datetime.strftime('%Y%m%d')
        print(base_day)
        year = str(ev.origins[0].time.datetime.year)
        yearday = ev.origins[0].time.datetime.strftime("%j")
        # Loop through stations to make the complete stream
        st = obspy.core.stream.Stream()
        for sta in stations:
            path = 'data/' + str(network) + '/' + year + '/' + yearday + '/' + sta + '.' + network + '.' + year + '.' + yearday
            tfile = tar.extractfile(path)
            stream = obspy.core.stream.read(tfile)
            st += stream
        st.resample(samp_rate)
    
    temp_cat = obspy.core.event.Catalog(events=[ev])

    # Process stream
    template_st = eqcorrscan.core.template_gen.template_gen(method="from_meta_file", st=st,lowcut=lowcut, highcut=highcut, samp_rate=samp_rate, length=length,
    filt_order=filt_order, prepick=prepick,meta_file=temp_cat, data_pad=data_pad,
    process_len=process_len, min_snr=min_snr, parallel=False,swin=swin,delayed=True,skip_short_chans=True)

    # Make template from processed waveform
    template = eqcorrscan.core.match_filter.template.Template(name=ev.resource_id.id, st=template_st[0], lowcut=lowcut, highcut=highcut, samp_rate=samp_rate, filt_order=filt_order, process_length=process_len, prepick=prepick, event=ev)
    template_list.append(template)

20170601


Pick for ENWF.HHE has no phase hint given, you should not use this template for cross-correlation re-picking!
Pick for ENWF.HHZ has no phase hint given, you should not use this template for cross-correlation re-picking!
Pick for KEMF.EHE has no phase hint given, you should not use this template for cross-correlation re-picking!
Pick for KEMF.EHZ has no phase hint given, you should not use this template for cross-correlation re-picking!
Pick for NCHR.EHE has no phase hint given, you should not use this template for cross-correlation re-picking!
Pick for NCHR.EHZ has no phase hint given, you should not use this template for cross-correlation re-picking!


AttributeError: 'NoneType' object has no attribute 'id'

In [ ]:
# Make a "tribe" from the templates
# Make sure that resource id in the tribe events matches the template
tribe = eqcorrscan.core.match_filter.tribe.Tribe(templates=template_list)
for i,t in enumerate(tribe):
    tribe[i].event.origins[0].resource_id = t.name

In [54]:
# Start with only a few events
filt_cat = cat[0:2]

template_list = []
for i,ev in enumerate(filt_cat):

    ###### Retrieve stream #####
    
    # Get year and year day
    year = str(ev.origins[0].time.datetime.year)
    yearday = ev.origins[0].time.datetime.strftime("%j")
    # Loop through stations
    st = obspy.core.stream.Stream()
    for sta in stations:
        path = 'data/' + str(network) + '/' + year + '/' + yearday + '/' + sta + '.' + network + '.' + year + '.' + yearday
        tfile = tar.extractfile(path)
        stream = obspy.core.stream.read(tfile)
        st += stream
    st.resample(samp_rate)
    
    
    temp_cat = obspy.core.event.Catalog(events=[ev])

    # Process stream
    template_st = eqcorrscan.core.template_gen.template_gen(method="from_meta_file", st=st,lowcut=lowcut, highcut=highcut, samp_rate=samp_rate, length=length,
    filt_order=filt_order, prepick=prepick,meta_file=temp_cat, data_pad=data_pad,
    process_len=process_len, min_snr=min_snr, parallel=False,swin=swin,delayed=True,skip_short_chans=True)

    # Make template from processed waveform
    template = eqcorrscan.core.match_filter.template.Template(name=ev.resource_id.id, st=template_st[0], lowcut=lowcut, highcut=highcut, samp_rate=samp_rate, filt_order=filt_order, process_length=process_len, prepick=prepick, event=ev)
    template_list.append(template)
    
tribe = eqcorrscan.core.match_filter.tribe.Tribe(templates=template_list)
for i,t in enumerate(tribe):
    tribe[i].event.origins[0].resource_id = t.name

In [ ]:
# Write templates to file, if desired
tribe.write('test_templates')

### Perform detection on miniseed data
To-Do: combining each day's detection into one Party

In [63]:
t1 = obspy.UTCDateTime(2017,6,1)
t2 = obspy.UTCDateTime(2017,6,3)
time_bins = np.arange(t1.datetime,t2.datetime,pd.Timedelta(1,'days'))
time_bins = [pd.to_datetime(t) for t in time_bins]

detections = []
for t in time_bins:
    t1 = obspy.UTCDateTime(t)
    t2 = obspy.UTCDateTime(t + pd.Timedelta(1,'day'))
    
     ###### Retrieve stream #####
    
    # Get year and year day
    year = str(t1.datetime.year)
    yearday = t1.datetime.strftime("%j")
    # Loop through stations
    st = obspy.core.stream.Stream()
    for sta in stations:
        path = 'data/' + str(network) + '/' + year + '/' + yearday + '/' + sta + '.' + network + '.' + year + '.' + yearday
        tfile = tar.extractfile(path)
        stream = obspy.core.stream.read(tfile)
        st += stream
    st.resample(samp_rate)
    

    stream = st
    det = tribe.detect(stream, threshold=threshold, threshold_type=threshold_type,trig_int=trig_int,parallel_process=False)
    detections.append(det)

In [69]:
test

Party of 2 Families.

In [70]:
test.write('test_quakeml.xml', format='quakeml')

Writing only the catalog component, metadata will not be preserved


Party of 2 Families.

In [71]:
det_cat = obspy.core.event.catalog.read_events('test_quakeml.xml')